In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "yelp_academic_dataset_business.json"  # or review, user, etc.

df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "yelp-dataset/yelp-dataset",
    file_path,
    pandas_kwargs={"lines": True}  # <-- Add this
)

print("First 5 records:", df.head())

First 5 records:               business_id                      name  \
0  Pns2l4eNsfO8kk83dixA6A  Abby Rappoport, LAC, CMQ   
1  mpf3x-BjTdTEA3yCZrAYPw             The UPS Store   
2  tUFrWirKiKi_TAnsVWINQQ                    Target   
3  MTSW4McQd7CbVtyjqoe9mw        St Honore Pastries   
4  mWMc6_wTdE0EUBKIGXDVfA  Perkiomen Valley Brewery   

                           address           city state postal_code  \
0           1616 Chapala St, Ste 2  Santa Barbara    CA       93101   
1  87 Grasso Plaza Shopping Center         Affton    MO       63123   
2             5255 E Broadway Blvd         Tucson    AZ       85711   
3                      935 Race St   Philadelphia    PA       19107   
4                    101 Walnut St     Green Lane    PA       18054   

    latitude   longitude  stars  review_count  is_open  \
0  34.426679 -119.711197    5.0             7        0   
1  38.551126  -90.335695    3.0            15        1   
2  32.223236 -110.880452    3.5            22      

In [3]:
# Filter
df_filtered = df[df['review_count'] > 5].copy()

print(f"Original: {len(df)} businesses")
print(f"After filtering: {len(df_filtered)} businesses")

Original: 150346 businesses
After filtering: 135425 businesses


In [4]:
import pandas as pd

# Replace nulls with empty dicts (keep existing dicts as-is)
df_filtered['attributes'] = df_filtered['attributes'].apply(lambda x: x if isinstance(x, dict) else {})
df_filtered['hours'] = df_filtered['hours'].apply(lambda x: x if isinstance(x, dict) else {})

# Flatten attributes into columns
attributes_df = pd.json_normalize(df_filtered['attributes'])
df_filtered = pd.concat([df_filtered.drop(columns=['attributes']), attributes_df], axis=1)

In [5]:
# Get primary category (first one listed)
df_filtered['primary_category'] = df_filtered['categories'].str.split(',').str[0]

# Or create a list of all categories
df_filtered['category_list'] = df_filtered['categories'].apply(
    lambda x: [c.strip() for c in x.split(',')] if pd.notna(x) else []
)

In [6]:
# Check distribution before deciding
print(df_filtered['review_count'].describe(percentiles=[.5, .9, .95, .99]))

# Optional: log-transform for modeling
import numpy as np
df_filtered['review_count_log'] = np.log1p(df_filtered['review_count'])

count    135425.000000
mean         49.259022
std         126.854359
min           6.000000
50%          17.000000
90%         107.000000
95%         188.000000
99%         499.000000
max        7568.000000
Name: review_count, dtype: float64


In [7]:
# Example: Business quality tiers
def classify_business(row):
    if row['stars'] >= 4.5 and row['review_count'] > 100:
        return 'High-Volume Premium'
    elif row['stars'] >= 4.0:
        return 'Quality Local'
    elif row['is_open'] == 0:
        return 'Closed Business'
    else:
        return 'Average/Mainstream'

df_filtered['persona_segment'] = df_filtered.apply(classify_business, axis=1)
print(df_filtered['persona_segment'].value_counts())

persona_segment
Average/Mainstream     66375
Quality Local          63252
Closed Business        15689
High-Volume Premium     3518
Name: count, dtype: int64


In [10]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

df_reviews = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "yelp-dataset/yelp-dataset",
    "yelp_academic_dataset_review.json",
    pandas_kwargs={"lines": True, "nrows": 500000}  # 500K reviews, loads in ~10 seconds
)

df_reviews_filtered = df_reviews[df_reviews['business_id'].isin(df_filtered['business_id'])]
print(f"Loaded {len(df_reviews_filtered):,} matching reviews")

Loaded 495,768 matching reviews


In [11]:
print("Loading businesses...")
df_business = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "yelp-dataset/yelp-dataset",
    "yelp_academic_dataset_business.json",
    pandas_kwargs={"lines": True}
)

# Filter: Only restaurants with > 5 reviews
df_restaurants = df_business[
    df_business['categories'].str.contains('Restaurants', case=False, na=False)
].copy()

df_filtered = df_restaurants[df_restaurants['review_count'] > 5].copy()

# Replace null attributes/hours with empty dicts (they're already dicts, not strings!)
df_filtered['attributes'] = df_filtered['attributes'].apply(lambda x: x if isinstance(x, dict) else {})
df_filtered['hours'] = df_filtered['hours'].apply(lambda x: x if isinstance(x, dict) else {})

# Extract primary category
df_filtered['primary_category'] = df_filtered['categories'].str.split(',').str[0]

print(f"Restaurants with >5 reviews: {len(df_filtered):,}")
print(df_filtered[['name', 'stars', 'review_count', 'primary_category']].head())

Loading businesses...
Restaurants with >5 reviews: 50,131
                     name  stars  review_count           primary_category
3      St Honore Pastries    4.0            80                Restaurants
5          Sonic Drive-In    2.0             6                    Burgers
8   Tsevi's Pub And Grill    3.0            19                       Pubs
9          Sonic Drive-In    1.5            10  Ice Cream & Frozen Yogurt
11  Vietnamese Food Truck    4.0            10                 Vietnamese


In [12]:
import json
import pandas as pd
from collections import defaultdict

# ── Step 1: Filter business dataset to restaurants only ──
restaurant_df = df[
    df['categories'].str.contains('Restaurant|Food', na=False) &
    (df['review_count'] >= 5) &        # min 5 reviews on the business
    (df['is_open'] == 1)               # only open businesses
].copy()

print(f"Restaurants after filter: {len(restaurant_df)}")
print(f"Cities: {restaurant_df['city'].value_counts().head(15)}")

Restaurants after filter: 44594
Cities: city
Philadelphia     4372
Tampa            2503
Indianapolis     2371
Tucson           2150
Nashville        2083
Edmonton         1935
New Orleans      1805
Saint Louis      1168
Reno             1139
Boise             782
Santa Barbara     673
Clearwater        555
Wilmington        535
St. Louis         496
Metairie          455
Name: count, dtype: int64


In [ ]:
import json
import numpy as np
import pandas as pd
from collections import defaultdict

# ── Step 1: Filter to 3 diaspora cities ──
TARGET_CITIES = ['Philadelphia', 'Tampa', 'Nashville', '']

city_restaurants = restaurant_df[
    restaurant_df['city'].isin(TARGET_CITIES)
].copy()

print(f"Total restaurants across 3 cities: {len(city_restaurants):,}")
print(city_restaurants['city'].value_counts())

# ── Step 2: Build restaurant_detail.csv ──
def is_halal(attrs):
    if not isinstance(attrs, dict):
        return False
    return 'halal' in str(attrs).lower()

def get_price(attrs):
    if not isinstance(attrs, dict):
        return '?'
    return attrs.get('RestaurantsPriceRange2', '?')

restaurant_detail = city_restaurants[[
    'business_id', 'name', 'city', 'stars', 'review_count',
    'categories', 'latitude', 'longitude'
]].copy()

restaurant_detail['halal'] = city_restaurants['attributes'].apply(is_halal)
restaurant_detail['price']  = city_restaurants['attributes'].apply(get_price)
restaurant_detail = restaurant_detail.set_index('business_id')

print(f"\nRestaurant detail shape: {restaurant_detail.shape}")
print(f"Halal restaurants: {restaurant_detail['halal'].sum()}")

restaurant_detail.to_csv('restaurant_detail.csv')
print("Saved restaurant_detail.csv ✓")

# ── Step 3: Load reviews for these restaurants ──
restaurant_ids = set(city_restaurants['business_id'].tolist())

df_reviews = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "yelp-dataset/yelp-dataset",
    "yelp_academic_dataset_review.json",
    pandas_kwargs={"lines": True, "nrows": 2000000}
)

df_reviews_cities = df_reviews[
    df_reviews['business_id'].isin(restaurant_ids)
].copy()

print(f"\nReviews for 3 cities: {len(df_reviews_cities):,}")

# ── Step 4: Filter active users (5+ reviews) ──
user_review_counts = df_reviews_cities.groupby('user_id').size()
active_user_ids    = user_review_counts[user_review_counts >= 5].index
df_active          = df_reviews_cities[df_reviews_cities['user_id'].isin(active_user_ids)].copy()

print(f"Active users (5+ reviews): {len(active_user_ids):,}")
print(f"Total reviews from active users: {len(df_active):,}")

# ── Step 5: Enrich reviews with restaurant metadata ──
df_enriched = df_active.merge(
    restaurant_detail[['name', 'city', 'categories', 'stars', 'price', 'halal']].reset_index(),
    on='business_id',
    suffixes=('_review', '_business')
)

print(f"\nEnriched reviews shape: {df_enriched.shape}")
print(f"Sample columns: {list(df_enriched.columns)}")

# ── Step 6: Build user history dict and save ──
def make_serializable(obj):
    if hasattr(obj, 'isoformat'):
        return obj.isoformat()
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.bool_):
        return bool(obj)
    raise TypeError(f"Not serializable: {type(obj)}")

user_history = {}
for user_id, group in df_enriched.groupby('user_id'):
    user_history[user_id] = group[[
        'business_id', 'name', 'city', 'categories',
        'stars_review', 'stars_business',
        'price', 'halal', 'text', 'date'
    ]].to_dict('records')

with open('user_review_history.json', 'w') as f:
    json.dump(user_history, f, default=make_serializable)

print(f"\nSaved {len(user_history):,} user histories ✓")



Total restaurants across 3 cities: 8,958
city
Philadelphia    4372
Tampa           2503
Nashville       2083
Name: count, dtype: int64

Restaurant detail shape: (8958, 9)
Halal restaurants: 1
Saved restaurant_detail.csv ✓


In [ ]:
# ── Step 7: Quick sanity check (fixed) ──
sample_uid  = list(user_history.keys())[0]
sample_revs = user_history[sample_uid]
print(f"\nSample user: {sample_uid}")
print(f"Number of reviews: {len(sample_revs)}")

# Convert to serializable for display only
sample = sample_revs[0].copy()
sample['date'] = str(sample['date'])
print(f"First review:\n{json.dumps(sample, indent=2)}")


Sample user: --KsuCSkGGvDKTbdK9NvIg
Number of reviews: 5
First review:
{
  "business_id": "ySn36gFeTJ0fR8CG-Q2DuA",
  "name": "Smokey Bones",
  "city": "Tampa",
  "categories": "Barbeque, Restaurants, Burgers, American (Traditional), Chicken Wings",
  "stars_review": 2,
  "stars_business": 3.5,
  "price": "2",
  "halal": false,
  "text": "Haven't been here in ages...and, after this most recent visit, I remember why. Mediocrity.\n\nThere were several new menu items that sounded appetizing, but they fell short when it came to taste. \n\nThe fried pickles were tasty, but, they're fried pickles, how could they not be good?\n\nThe corn dogs were cooked in their doughnut batter. My husband didn't like them at all; I kind of enjoyed the sweetness, but not enough to eat a whole one.\n\nThe fries weren't good at all. So disappointing because I used to enjoy their fries. \n\nAll in all the food was mediocre. I can't imagine I'll be returning anytime soon.",
  "date": "2013-11-16 01:15:12"
}


In [ ]:
import json
import numpy as np

# ── Fix: convert dates to strings BEFORE building the dict ──
df_enriched['date'] = df_enriched['date'].astype(str)

# ── Rebuild user history ──
user_history = {}
for user_id, group in df_enriched.groupby('user_id'):
    user_history[user_id] = group[[
        'business_id', 'name', 'city', 'categories',
        'stars_review', 'stars_business',
        'price', 'halal', 'text', 'date'
    ]].to_dict('records')

# ── Serializer for any remaining numpy types ──
def make_serializable(obj):
    if isinstance(obj, np.integer): return int(obj)
    if isinstance(obj, np.floating): return float(obj)
    if isinstance(obj, np.bool_): return bool(obj)
    raise TypeError(f"Not serializable: {type(obj)}")

# ── Save ──
with open('/kaggle/working/user_review_history.json', 'w') as f:
    json.dump(user_history, f, default=make_serializable)

print(f"✓ Saved {len(user_history):,} user histories")

# ── Verify ──
sample_uid = list(user_history.keys())[0]
print(f"\nSample user: {sample_uid}")
print(f"Reviews: {len(user_history[sample_uid])}")
print(json.dumps(user_history[sample_uid][0], indent=2))

✓ Saved 9,614 user histories

Sample user: --KsuCSkGGvDKTbdK9NvIg
Reviews: 5
{
  "business_id": "ySn36gFeTJ0fR8CG-Q2DuA",
  "name": "Smokey Bones",
  "city": "Tampa",
  "categories": "Barbeque, Restaurants, Burgers, American (Traditional), Chicken Wings",
  "stars_review": 2,
  "stars_business": 3.5,
  "price": "2",
  "halal": false,
  "text": "Haven't been here in ages...and, after this most recent visit, I remember why. Mediocrity.\n\nThere were several new menu items that sounded appetizing, but they fell short when it came to taste. \n\nThe fried pickles were tasty, but, they're fried pickles, how could they not be good?\n\nThe corn dogs were cooked in their doughnut batter. My husband didn't like them at all; I kind of enjoyed the sweetness, but not enough to eat a whole one.\n\nThe fries weren't good at all. So disappointing because I used to enjoy their fries. \n\nAll in all the food was mediocre. I can't imagine I'll be returning anytime soon.",
  "date": "2013-11-16 01:15:12"


In [ ]:
!pip install -q langchain langchain-google-genai google-generativeai tqdm "openai>=1.0"


In [ ]:
!pip install -q langchain-groq

In [ ]:
import os
import json
import random
import time
import re
from collections import Counter
from pathlib import Path
from types import SimpleNamespace

import pandas as pd
from langchain_core.messages import HumanMessage, SystemMessage
from tqdm import tqdm

# Optional providers. Pick one below with LLM_PROVIDER.
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from openai import OpenAI

# Ethnic priors are intentionally broad candidate attributes, not substitutes for history.
ETHNIC_GROUPS = {
    "yoruba": {
        "weight": 0.40,
        "spice_tolerance": "very high",
        "food_values": (
            "Yoruba Nigerian living in the diaspora; often values bold pepper, stews, "
            "West African food, communal dining, and expressive food talk."
        ),
        "language_hints": "Natural Yoruba-Nigerian/Pidgin expressions may include omo, e jo, shebi, abi, jare.",
        "review_style": "expressive and opinionated, especially about spice and flavour",
        "halal_sensitive": False,
    },
    "igbo": {
        "weight": 0.35,
        "spice_tolerance": "moderate",
        "food_values": (
            "Igbo Nigerian living in the diaspora; often values rich soups, generous portions, "
            "ingredient quality, loyalty to good restaurants, and value for money."
        ),
        "language_hints": "Natural Igbo-Nigerian/Pidgin expressions may include nna, biko, chai, ndo, oga.",
        "review_style": "practical, value-focused, specific about portions and quality",
        "halal_sensitive": False,
    },
    "hausa": {
        "weight": 0.25,
        "spice_tolerance": "low to moderate",
        "food_values": (
            "Hausa Nigerian living in the diaspora; often values halal options, grilled meats, "
            "savory depth, warm service, and Muslim-friendly restaurants."
        ),
        "language_hints": "Natural Hausa-Nigerian/Pidgin expressions may include wallahi, kai, sannu, haba.",
        "review_style": "calm, service-focused, attentive to halal status and hospitality",
        "halal_sensitive": True,
    },
}

# SimUSER + Agent4Rec-style extraction settings.
METHOD_ID = "simusers_selfconsistent_persona_v3"
OUTPUT_DIR = Path('/kaggle/working/simulation_simusers_v3')
# Speed knobs:
# - "quick": 2 LLM calls/user, no self-consistency scoring. Good for checking outputs.
# - "balanced": ~7 LLM calls/user, light self-consistency scoring.
# - "full": ~17 LLM calls/user, closer to SimUSER paper. Use for final generation.
RUN_PRESET = "quick"

PRESETS = {
    "quick": {"n_users": 10, "self_consistency": False, "scoring_subsets": 0, "scoring_subset_size": 5},
    "balanced": {"n_users": 100, "self_consistency": True, "scoring_subsets": 1, "scoring_subset_size": 6},
    "full": {"n_users": 1000, "self_consistency": True, "scoring_subsets": 3, "scoring_subset_size": 8},
}

preset = PRESETS[RUN_PRESET]
N_USERS = preset["n_users"]
MIN_REVIEWS = 5
SEED = 42
HISTORY_SAMPLE_SIZE = 50
CANDIDATE_PERSONAS = 5
LIKE_THRESHOLD = 4       # SimUSER: liked if rating >= 4
DISLIKE_THRESHOLD = 3    # SimUSER extraction: disliked if rating < 3; rating == 3 is neutral/ignored
MEMORY_DISLIKE_MAX = 2   # SimUSER memory appendix: scores <= 2 are disliked
RUN_SELF_CONSISTENCY = preset["self_consistency"]
SCORING_SUBSETS = preset["scoring_subsets"]
SCORING_SUBSET_SIZE = preset["scoring_subset_size"]
LLM_PROVIDER = "openai"  # "openai", "groq", or "gemini"
OPENAI_MODEL = "o4-mini"
OPENAI_REASONING_EFFORT = "low"  # use "medium" for final full-quality runs
OPENAI_MAX_OUTPUT_TOKENS = 4000

POSSIBLE_AGES = ["18-24", "25-34", "35-44", "45-54", "55+"]
POSSIBLE_OCCUPATIONS = [
    "student", "healthcare worker", "engineer", "teacher", "small business owner",
    "hospitality worker", "delivery/logistics worker", "office administrator", "creative worker",
]
BIG_FIVE_TRAITS = ["openness", "conscientiousness", "extraversion", "agreeableness", "neuroticism"]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
rng = random.Random(SEED)

# Load data.
with open('/kaggle/working/user_review_history.json') as f:
    user_review_history = json.load(f)
restaurant_detail = pd.read_csv('/kaggle/working/restaurant_detail.csv', index_col='business_id')

qualified = {uid: revs for uid, revs in user_review_history.items() if len(revs) >= MIN_REVIEWS}
selected = rng.sample(list(qualified.keys()), min(N_USERS, len(qualified)))
print(f"Qualified users: {len(qualified):,}")
print(f"Selected users: {len(selected):,}")
print(f"Run preset: {RUN_PRESET} | self_consistency={RUN_SELF_CONSISTENCY} | scoring_subsets={SCORING_SUBSETS}")
print(f"LLM provider: {LLM_PROVIDER}" + (f" | model={OPENAI_MODEL} | reasoning_effort={OPENAI_REASONING_EFFORT}" if LLM_PROVIDER == "openai" else ""))


def assign_ethnic_groups(user_ids, seed=42):
    local_rng = random.Random(seed)
    groups = list(ETHNIC_GROUPS.keys())
    weights = [ETHNIC_GROUPS[g]["weight"] for g in groups]
    return {uid: local_rng.choices(groups, weights=weights, k=1)[0] for uid in user_ids}


ethnic_assignments = assign_ethnic_groups(selected, seed=SEED)
print(f"Ethnic distribution: {dict(Counter(ethnic_assignments.values()))}")


class OpenAIResponsesLLM:
    def __init__(self, model=OPENAI_MODEL, reasoning_effort=OPENAI_REASONING_EFFORT, max_output_tokens=OPENAI_MAX_OUTPUT_TOKENS):
        self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        self.model = model
        self.reasoning_effort = reasoning_effort
        self.max_output_tokens = max_output_tokens

    def _message_content(self, message):
        return getattr(message, "content", str(message))

    def invoke(self, messages):
        # OpenAI reasoning models are called through the Responses API.
        # Combine system/developer instructions and user content into one user input so
        # this wrapper can accept the same LangChain message objects used elsewhere.
        prompt_parts = []
        for message in messages:
            role_name = message.__class__.__name__.replace("Message", "").upper()
            prompt_parts.append(f"{role_name}: {self._message_content(message)}")
        response = self.client.responses.create(
            model=self.model,
            reasoning={"effort": self.reasoning_effort},
            input=[{"role": "user", "content": "\n\n".join(prompt_parts)}],
            max_output_tokens=self.max_output_tokens,
        )
        return SimpleNamespace(content=response.output_text or "")


def make_llm(provider=LLM_PROVIDER):
    if provider == "openai":
        if not os.getenv("OPENAI_API_KEY"):
            raise ValueError("Set OPENAI_API_KEY before running persona generation.")
        return OpenAIResponsesLLM()

    if provider == "groq":
        api_key = os.getenv("GROQ_API_KEY")
        if not api_key:
            raise ValueError("Set GROQ_API_KEY before running persona generation.")
        return ChatGroq(
            model="llama-3.1-8b-instant",
            groq_api_key=api_key,
            temperature=0.75,
        )

    api_key = os.getenv("GOOGLE_API_KEY")
    if not api_key:
        raise ValueError("Set GOOGLE_API_KEY before running persona generation.")
    return ChatGoogleGenerativeAI(
        model="gemini-1.5-flash",
        google_api_key=api_key,
        temperature=0.75,
        convert_system_message_to_human=True,
    )


llm = make_llm()


def normalize_response_content(response):
    content = response.content
    if isinstance(content, list):
        content = " ".join(
            block.get("text", "") if isinstance(block, dict) else str(block)
            for block in content
        )
    return str(content).strip()


def sample_user_history(user_reviews, sample_size=HISTORY_SAMPLE_SIZE, seed_key=""):
    # Deterministic per user, but still random rather than top-rating-biased.
    local_rng = random.Random(f"{SEED}:{seed_key}")
    reviews = list(user_reviews)
    if len(reviews) <= sample_size:
        local_rng.shuffle(reviews)
        return reviews
    return local_rng.sample(reviews, sample_size)


def rating_label(stars):
    stars = float(stars)
    if stars >= LIKE_THRESHOLD:
        return "LIKED"
    if stars < DISLIKE_THRESHOLD:
        return "DISLIKED"
    return "NEUTRAL"


def memory_label(stars):
    stars = float(stars)
    if stars >= LIKE_THRESHOLD:
        return "liked"
    if stars <= MEMORY_DISLIKE_MAX:
        return "disliked"
    return "neutral"


def category_tokens(categories):
    return [c.strip().lower() for c in str(categories or "").split(',') if c.strip()]


def quantile_bucket(value, values, labels=("low", "medium", "high")):
    values = sorted(float(v) for v in values if pd.notna(v))
    if not values:
        return labels[1]
    q1 = values[int((len(values) - 1) * 0.33)]
    q2 = values[int((len(values) - 1) * 0.67)]
    if value <= q1:
        return labels[0]
    if value <= q2:
        return labels[1]
    return labels[2]


def raw_user_statistics(reviews):
    ratings = [float(r["stars_review"]) for r in reviews]
    avg_rating = sum(ratings) / len(ratings)
    conformity = sum((float(r["stars_review"]) - float(r.get("stars_business", r["stars_review"]))) ** 2 for r in reviews) / len(reviews)
    categories = set()
    for r in reviews:
        categories.update(category_tokens(r.get("categories", "")))
    return {
        "n_reviews": len(reviews),
        "avg_rating": avg_rating,
        "conformity_raw": conformity,
        "variety_raw": len(categories),
    }


raw_stats_by_user = {uid: raw_user_statistics(revs) for uid, revs in qualified.items()}
activity_values = [v["n_reviews"] for v in raw_stats_by_user.values()]
conformity_values = [v["conformity_raw"] for v in raw_stats_by_user.values()]
variety_values = [v["variety_raw"] for v in raw_stats_by_user.values()]


def pickiness_from_average(avg_rating):
    if avg_rating >= 4.5:
        return "not picky"
    if avg_rating >= 3.5:
        return "moderately picky"
    return "extremely picky"


def user_behavior_traits(uid):
    stats = raw_stats_by_user[uid]
    # Higher squared deviation means lower conformity to restaurant-average ratings.
    conformity_level = quantile_bucket(stats["conformity_raw"], conformity_values, labels=("high", "medium", "low"))
    return {
        "pickiness": pickiness_from_average(stats["avg_rating"]),
        "engagement": quantile_bucket(stats["n_reviews"], activity_values),
        "conformity": conformity_level,
        "variety": quantile_bucket(stats["variety_raw"], variety_values),
        **stats,
    }


def format_review_evidence(reviews, include_neutral=False):
    lines = []
    for r in reviews:
        label = rating_label(r["stars_review"])
        if label == "NEUTRAL" and not include_neutral:
            continue
        halal_tag = " [HALAL]" if r.get("halal") else ""
        review_text = re.sub(r"\s+", " ", str(r.get("text", ""))).strip()[:180]
        lines.append(
            f'- {label}: "{r["name"]}" | {str(r.get("categories", ""))[:80]}{halal_tag} '
            f'| price={r.get("price", "?")} | user_rating={r["stars_review"]} | '
            f'business_stars={r.get("stars_business", "?")} | review="{review_text}"'
        )
    return "\n".join(lines)


def build_memory_entries(reviews):
    entries = []
    for r in reviews:
        label = memory_label(r["stars_review"])
        if label == "liked":
            entries.append(f'I liked {r["name"]} based on my review score of {r["stars_review"]}.')
        elif label == "disliked":
            entries.append(f'I disliked {r["name"]} based on my review score of {r["stars_review"]}.')
        else:
            entries.append(f'I felt neutral about {r["name"]} based on my review score of {r["stars_review"]}.')
    return "\n".join(entries)


def build_summary_prompt(evidence):
    return f"""Summarize this user's restaurant preferences from the evidence.

Use only the evidence. Distinguish LIKED items from DISLIKED items. Mention rating patterns, cuisines, service/ambience, price/value, dietary constraints, and repeated review-language signals. Do not invent demographics.

Evidence:
{evidence}

Return 5-8 concise bullets."""


def generate_preference_summary(llm, evidence, retries=3):
    for attempt in range(retries):
        try:
            response = llm.invoke([
                SystemMessage(content="You extract faithful, evidence-grounded user preference summaries."),
                HumanMessage(content=build_summary_prompt(evidence)),
            ])
            return normalize_response_content(response)
        except Exception as e:
            print(f"  Summary error (attempt {attempt + 1}): {e}")
            time.sleep(5 * (attempt + 1))
    return "- Preference summary unavailable; use the labeled evidence only."


def build_persona_system_prompt(ethnic_group):
    grp = ETHNIC_GROUPS[ethnic_group]
    halal_rule = "Include halal sensitivity only if supported by the evidence or candidate background." if grp["halal_sensitive"] else "Do not force halal sensitivity unless the evidence supports it."
    return f"""You generate candidate recommender-system personas that match restaurant interaction histories.

Candidate cultural background: {ethnic_group} Nigerian diaspora.
Cultural hint: {grp['food_values']}
Language hint: {grp['language_hints']}
Review style hint: {grp['review_style']}
{halal_rule}

Important constraints:
- The persona must be consistent with the preference summary, liked/disliked evidence, pickiness, and behavioral traits.
- Use demographic attributes as light hypotheses, not stereotypes.
- Do not include names or gender.
- AGE must be exactly one of the allowed age ranges provided by the user. Do not invent exact ages.
- OCCUPATION must be exactly one of the allowed occupations provided by the user.
- Big Five traits must be integers from 1 to 3.
- Generate exactly {CANDIDATE_PERSONAS} distinct candidate personas.
- Each candidate must include all fields in the requested format.
- Keep each field to one concise line. Do not use markdown bullets, headings, or raw review excerpts.
- EVIDENCE RESTAURANTS must list only restaurant names, separated by semicolons, with at most 8 names.
"""


def build_persona_user_prompt(preference_summary, evidence, behavior_traits):
    return f"""Preference summary su:
{preference_summary}

Historical evidence:
{evidence}

Computed user attributes from historical data:
PICKINESS: {behavior_traits['pickiness']}
ENGAGEMENT: {behavior_traits['engagement']} (n_reviews={behavior_traits['n_reviews']})
CONFORMITY: {behavior_traits['conformity']} (rating deviation={behavior_traits['conformity_raw']:.3f})
VARIETY: {behavior_traits['variety']} (unique category count={behavior_traits['variety_raw']})
AVERAGE RATING: {behavior_traits['avg_rating']:.2f}

Allowed ages, choose exactly one: {', '.join(POSSIBLE_AGES)}
Allowed occupations, choose exactly one: {', '.join(POSSIBLE_OCCUPATIONS)}
Big Five traits: {', '.join(BIG_FIVE_TRAITS)}; each must be scored 1, 2, or 3.

Generate {CANDIDATE_PERSONAS} candidate personas in this exact format:

CANDIDATE 1
AGE: [one allowed age range only]
OCCUPATION: [one allowed occupation only]
BIG FIVE: openness=..., conscientiousness=..., extraversion=..., agreeableness=..., neuroticism=...
PICKINESS: ...
ENGAGEMENT: ...
CONFORMITY: ...
VARIETY: ...
PREFERENCE SUMMARY: ...
LIKES: ...
DISLIKES: ...
RATING TENDENCY: ...
EVIDENCE RESTAURANTS: ...

CANDIDATE 2
..."""


def count_candidate_blocks(persona_text):
    return len(re.findall(r'(?im)^\s*CANDIDATE\s+\d+\b', persona_text or ""))


def candidate_ages_are_allowed(persona_text):
    ages = re.findall(r'(?im)^\s*AGE\s*:\s*(.+?)\s*$', persona_text or "")
    return len(ages) >= CANDIDATE_PERSONAS and all(age.strip() in POSSIBLE_AGES for age in ages[:CANDIDATE_PERSONAS])


def generate_candidate_personas(llm, system_prompt, user_prompt, retries=3):
    last_content = ""
    for attempt in range(retries):
        try:
            response = llm.invoke([
                SystemMessage(content=system_prompt),
                HumanMessage(content=user_prompt),
            ])
            content = normalize_response_content(response)
            last_content = content
            if count_candidate_blocks(content) >= CANDIDATE_PERSONAS and candidate_ages_are_allowed(content):
                return content
            print(
                f"  Persona format retry {attempt + 1}: "
                f"candidates={count_candidate_blocks(content)}, allowed_ages={candidate_ages_are_allowed(content)}"
            )
            user_prompt = user_prompt + (
                "\n\nFORMAT CORRECTION: Regenerate exactly 5 candidates. "
                "Every AGE must be one of the allowed age ranges exactly, not an exact number. "
                "Keep every field one line and do not include raw review excerpts."
            )
        except Exception as e:
            print(f"  Persona error (attempt {attempt + 1}): {e}")
        time.sleep(5 * (attempt + 1))
    return last_content


def split_candidate_personas(persona_text):
    matches = list(re.finditer(r'(?im)^\s*CANDIDATE\s+(\d+)\b', persona_text))
    if not matches:
        return [persona_text.strip()] if persona_text.strip() else []
    candidates = []
    for i, match in enumerate(matches):
        start = match.start()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(persona_text)
        candidates.append(persona_text[start:end].strip())
    return candidates


def compact_persona_text(persona_text):
    required = [
        "CANDIDATE", "AGE", "OCCUPATION", "BIG FIVE", "PICKINESS", "ENGAGEMENT",
        "CONFORMITY", "VARIETY", "PREFERENCE SUMMARY", "LIKES", "DISLIKES",
        "RATING TENDENCY", "EVIDENCE RESTAURANTS",
    ]
    lines = []
    current_key = None
    for raw_line in persona_text.splitlines():
        line = raw_line.strip().lstrip('-+* ').strip()
        if not line:
            continue
        key_match = re.match(r'^(CANDIDATE\s+\d+|[A-Z ]+):?\s*(.*)$', line)
        if key_match:
            key = key_match.group(1).strip()
            value = key_match.group(2).strip()
            if key.startswith("CANDIDATE"):
                lines.append(key)
                current_key = "CANDIDATE"
            elif key in required:
                if key == "EVIDENCE RESTAURANTS":
                    names = [n.strip().strip('"') for n in re.split(r';|,', value) if n.strip()]
                    value = '; '.join(dict.fromkeys(names[:8]))
                lines.append(f"{key}: {value[:500]}")
                current_key = key
            continue
        # Append short continuation text only for summary-like fields; drop raw evidence dumps.
        if current_key in {"PREFERENCE SUMMARY", "LIKES", "DISLIKES", "RATING TENDENCY"} and lines:
            lines[-1] = (lines[-1] + " " + line)[:600]
    return "\n".join(lines).strip()


def parse_score_pair(text):
    target = re.search(r'TARGET_SCORE\s*:\s*([1-5])', text, flags=re.I)
    other = re.search(r'OTHER_SCORE\s*:\s*([1-5])', text, flags=re.I)
    if target and other:
        return int(target.group(1)), int(other.group(1))
    nums = [int(n) for n in re.findall(r'\b[1-5]\b', text)]
    if len(nums) >= 2:
        return nums[0], nums[1]
    return 3, 3


def sample_subset(reviews, size, rng_obj):
    reviews = list(reviews)
    if len(reviews) <= size:
        rng_obj.shuffle(reviews)
        return reviews
    return rng_obj.sample(reviews, size)


def score_candidate_persona(llm, candidate, target_uid, all_user_ids, subset_size=SCORING_SUBSET_SIZE, n_subsets=SCORING_SUBSETS):
    local_rng = random.Random(f"{SEED}:score:{target_uid}:{candidate[:80]}")
    score = 0
    details = []
    target_reviews_all = user_review_history[target_uid]
    other_pool = [uid for uid in all_user_ids if uid != target_uid and len(user_review_history[uid]) >= subset_size]

    for round_idx in range(n_subsets):
        if not other_pool:
            break
        other_uid = local_rng.choice(other_pool)
        target_subset = sample_subset(target_reviews_all, subset_size, local_rng)
        other_subset = sample_subset(user_review_history[other_uid], subset_size, local_rng)
        target_evidence = format_review_evidence(target_subset, include_neutral=True)
        other_evidence = format_review_evidence(other_subset, include_neutral=True)
        prompt = f"""Persona:
{candidate}

Rate how well the persona explains each interaction subset. Use evidence only.

TARGET USER SUBSET:
{target_evidence}

OTHER USER SUBSET:
{other_evidence}

Return exactly:
TARGET_SCORE: integer 1-5
OTHER_SCORE: integer 1-5
REASON: one short sentence"""
        try:
            response = llm.invoke([
                SystemMessage(content="You are scoring persona-history consistency for recommender-system user simulation."),
                HumanMessage(content=prompt),
            ])
            text = normalize_response_content(response)
            target_score, other_score = parse_score_pair(text)
        except Exception as e:
            print(f"  Scoring error for user {target_uid} round {round_idx + 1}: {e}")
            target_score, other_score, text = 3, 3, "scoring failed"
        score += target_score - other_score
        details.append({
            "round": round_idx + 1,
            "other_user_id": other_uid,
            "target_score": target_score,
            "other_score": other_score,
            "delta": target_score - other_score,
        })
        time.sleep(1)
    return score, details


def select_best_persona(llm, persona_text, target_uid, all_user_ids):
    candidates = split_candidate_personas(persona_text)
    if not candidates:
        return "", [], []
    if not RUN_SELF_CONSISTENCY:
        first = candidates[0]
        return first, [{"candidate_id": 1, "score": None, "details": [], "text": first}], candidates

    scored = []
    for idx, candidate in enumerate(candidates[:CANDIDATE_PERSONAS], start=1):
        score, details = score_candidate_persona(llm, candidate, target_uid, all_user_ids)
        scored.append({"candidate_id": idx, "score": score, "details": details, "text": candidate})
    best = max(scored, key=lambda row: row["score"])
    return best["text"], scored, candidates


user_index_map = {}
index_path = OUTPUT_DIR / 'users.json'
if index_path.exists():
    with open(index_path) as f:
        user_index_map = json.load(f)
    print(f"Resuming from checkpoint: {len(user_index_map)} users already done")

already_done = set(user_index_map.keys())
all_qualified_ids = list(qualified.keys())

for idx, uid in enumerate(tqdm(selected)):
    str_idx = str(idx)
    if str_idx in already_done:
        continue

    sampled_reviews = sample_user_history(user_review_history[uid], seed_key=uid)
    evidence = format_review_evidence(sampled_reviews)
    if not evidence.strip():
        evidence = format_review_evidence(sampled_reviews, include_neutral=True)
    if not evidence.strip():
        continue

    ethnic_group = ethnic_assignments[uid]
    behavior_traits = user_behavior_traits(uid)
    preference_summary = generate_preference_summary(llm, evidence)
    candidate_text = generate_candidate_personas(
        llm,
        build_persona_system_prompt(ethnic_group),
        build_persona_user_prompt(preference_summary, evidence, behavior_traits),
    )
    if not candidate_text.strip():
        continue

    selected_persona, candidate_scores, candidates = select_best_persona(llm, candidate_text, uid, all_qualified_ids)
    if not selected_persona.strip():
        selected_persona = candidates[0] if candidates else candidate_text
    selected_persona = compact_persona_text(selected_persona)

    memory_entries = build_memory_entries(user_review_history[uid])

    with open(OUTPUT_DIR / f"persona_{idx}.txt", 'w', encoding='utf-8') as f:
        f.write(selected_persona)
    with open(OUTPUT_DIR / f"candidate_personas_{idx}.txt", 'w', encoding='utf-8') as f:
        f.write(candidate_text)
    with open(OUTPUT_DIR / f"preference_summary_{idx}.txt", 'w', encoding='utf-8') as f:
        f.write(preference_summary)
    with open(OUTPUT_DIR / f"evidence_{idx}.txt", 'w', encoding='utf-8') as f:
        f.write(evidence)
    with open(OUTPUT_DIR / f"memory_{idx}.txt", 'w', encoding='utf-8') as f:
        f.write(memory_entries)
    with open(OUTPUT_DIR / f"candidate_scores_{idx}.json", 'w', encoding='utf-8') as f:
        json.dump([{k: v for k, v in row.items() if k != "text"} for row in candidate_scores], f, indent=2)

    label_counts = Counter(rating_label(r["stars_review"]) for r in sampled_reviews)
    numeric_scores = [row["score"] for row in candidate_scores if row.get("score") is not None]
    best_score = max(numeric_scores) if numeric_scores else None
    best_candidate_id = (
        max(candidate_scores, key=lambda row: row["score"] if row.get("score") is not None else -999)["candidate_id"]
        if candidate_scores else 1
    )
    user_index_map[str_idx] = {
        "yelp_user_id": uid,
        "ethnic_group": ethnic_group,
        "n_reviews": len(user_review_history[uid]),
        "sampled_reviews": len(sampled_reviews),
        "liked_sampled": label_counts.get("LIKED", 0),
        "disliked_sampled": label_counts.get("DISLIKED", 0),
        "neutral_sampled": label_counts.get("NEUTRAL", 0),
        "persona_candidates": len(candidates),
        "selected_candidate_id": best_candidate_id,
        "selected_candidate_score": best_score,
        "run_preset": RUN_PRESET,
        "self_consistency": RUN_SELF_CONSISTENCY,
        "method": METHOD_ID,
        **behavior_traits,
    }

    with open(index_path, 'w', encoding='utf-8') as f:
        json.dump(user_index_map, f, indent=2)

    time.sleep(4)

print(f"\nSaved {len(user_index_map)} users to {OUTPUT_DIR}")
with open(OUTPUT_DIR / 'persona_0.txt', encoding='utf-8') as f:
    print(f.read()[:800])


Qualified users: 9,614

Ethnic distribution: {'igbo': 369, 'yoruba': 374, 'hausa': 257}
Resuming from checkpoint: 393 personas already done


 49%|████▉     | 492/1000 [24:39<1:55:12, 13.61s/it]

In [ ]:
# Debug: inspect the SimUSER-style evidence, traits, and memory for one selected user.
sample_uid = selected[0]
sample_reviews = sample_user_history(user_review_history[sample_uid], seed_key=sample_uid)
evidence = format_review_evidence(sample_reviews)
if not evidence.strip():
    evidence = format_review_evidence(sample_reviews, include_neutral=True)
traits = user_behavior_traits(sample_uid)

print(f"User: {sample_uid}")
print(f"Total reviews: {len(user_review_history[sample_uid])}")
print(f"Sampled reviews: {len(sample_reviews)}")
print(Counter(rating_label(r["stars_review"]) for r in sample_reviews))
print(f"Traits: {traits}")
print(f"\nEvidence length: {len(evidence)}")
print(f"Evidence preview:\n{evidence[:1000]}")
print(f"\nMemory preview:\n{build_memory_entries(user_review_history[sample_uid])[:1000]}")


In [ ]:
# Preview selected personas and their self-consistency scores.
for i in range(5):
    path = OUTPUT_DIR / f"persona_{i}.txt"
    score_path = OUTPUT_DIR / f"candidate_scores_{i}.json"
    if path.exists():
        meta = user_index_map.get(str(i), {})
        print(f"\n{'='*50}")
        print(
            f"USER {i} SELECTED PERSONA — {meta.get('ethnic_group', 'unknown').upper()} "
            f"candidate={meta.get('selected_candidate_id')} score={meta.get('selected_candidate_score')}"
        )
        print(f"{'='*50}")
        print(open(path, encoding='utf-8').read())
        if score_path.exists():
            print("\nCandidate scores:")
            print(open(score_path, encoding='utf-8').read())


In [ ]:
# Run this RIGHT NOW before doing anything else
import shutil
import os

# Check what's already saved
saved_personas = list(OUTPUT_DIR.glob('persona_*.txt'))
print(f"Personas on disk: {len(saved_personas)}")

# Verify users.json checkpoint
if (OUTPUT_DIR / 'users.json').exists():
    with open(OUTPUT_DIR / 'users.json') as f:
        checkpoint = json.load(f)
    print(f"Checkpoint has: {len(checkpoint)} entries")
else:
    print("No checkpoint found!")


In [ ]:
import json
from pathlib import Path

OUTPUT_DIR = Path('/kaggle/working/simulation_simusers_v3')

# Count saved files
persona_files = list(OUTPUT_DIR.glob('persona_*.txt'))
print(f"Persona files on disk: {len(persona_files)}")

# Check checkpoint
if (OUTPUT_DIR / 'users.json').exists():
    with open(OUTPUT_DIR / 'users.json') as f:
        checkpoint = json.load(f)
    print(f"Checkpoint entries: {len(checkpoint)}")
    print(f"Ethnic breakdown: {{}}")
    from collections import Counter
    print(Counter(v['ethnic_group'] for v in checkpoint.values()))
else:
    print("No checkpoint found")


In [ ]:
def make_llm(provider=LLM_PROVIDER):
    if provider == "openai":
        return OpenAIResponsesLLM()

    if provider == "openrouter":
        client = OpenAI(
            api_key=OPENROUTER_API_KEY,
            base_url="https://openrouter.ai/api/v1",
        )
        class OpenRouterLLM:
            def invoke(self, messages):
                formatted = []
                for m in messages:
                    role = "system" if m.__class__.__name__ == "SystemMessage" else "user"
                    formatted.append({"role": role, "content": m.content})
                resp = client.chat.completions.create(
                    model="inclusionai/ring-2.6-1t:free",
                    messages=formatted,
                    temperature=0.75,
                    max_tokens=4000,
                )
                from types import SimpleNamespace
                return SimpleNamespace(content=resp.choices[0].message.content or "")
        return OpenRouterLLM()

    if provider == "groq":

In [ ]:
class OpenRouterLLM:
            def invoke(self_, messages):
                formatted = []
                for m in messages:
                    msg_role = "system" if m.__class__.__name__ == "SystemMessage" else "user"
                    formatted.append({"role": msg_role, "content": m.content})
                resp = client.chat.completions.create(
                    model="inclusionai/ring-2.6-1t:free",
                    messages=formatted,
                    temperature=0.75,
                    max_tokens=4000,
                )
                from types import SimpleNamespace
                return SimpleNamespace(content=resp.choices[0].message.content or "")
        return OpenRouterLLM()